In [1]:
pip install -qU weaviate-client transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 972.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 583.8/583.8 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 1.4 MB/s eta 0:00:00


#Connect to Weaviate Cloud

In [71]:
import weaviate
import requests
import json
from google.colab import userdata
from weaviate.classes.init import Auth
from weaviate.classes.query import Rerank
from transformers import (
    AutoTokenizer,
    pipeline
)

def connect_to_db():
    headers = {
        "X-Cohere-Api-Key": userdata.get('CohereRerank')
    }

    # Best practice: store your credentials in environment variables
    weaviate_url = userdata.get('veaviat_rest')
    weaviate_api_key = userdata.get('weaviat_api_key')

    # Connect to Weaviate Cloud
    client = weaviate.connect_to_weaviate_cloud(
        cluster_url=weaviate_url,
        auth_credentials=Auth.api_key(weaviate_api_key),
        headers=headers
    )

    print(client.is_ready())  # Should print: `True`

    #client.close()  # Free up resources
    return client

In [53]:
def search_for_faq(user_query, client):
    collection = client.collections.use("FAQ")

    query = f"{user_query}".strip()

    response = collection.query.hybrid(
        query=query,  # The model provider integration will automatically vectorize the query
        limit=10,
        alpha = 0.40,
        rerank=Rerank(
            prop="content",  # الخاصية المراد إعادة ترتيبها بناءً عليها
            query=query,  # يمكن أن يكون مختلف عن الاستعلام الأصلي
        )
    )

    queries = []

    for idx, obj in enumerate(response.objects[:2]):
        message_to_router = f"{query} [SEP] {obj.properties["content"]}"
        queries.append(message_to_router)

    #bert_query = f"{query} [SEP] {response.objects[0].properties["content"]}"

    return queries, response.objects[:3]

In [127]:
def router_decision(queries):
  model_checkpoint = "EN3IMI/RouterAraBERT"
  tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

  classifier = pipeline("sentiment-analysis", model=model_checkpoint)

  results = classifier(queries)
  labels = [res['label'] for res in results]
  numeric_labels = [1 if label == 'LABEL_1' else 0 for label in labels]

  return any(numeric_labels)

In [115]:
def llm_response(query, docs):
  chunks = []
  for i in docs:
    chunks.append(i.properties["content"])

  API_KEY = userdata.get('preplexity_api_key')  # تأكد من تعيين المفتاح في البيئة
  ENDPOINT = "https://api.perplexity.ai/chat/completions"

  # system prompt يوضح للموديل انه يعتمد فقط على المعلومات المعطاة
  system_prompt = """
  You are an intelligent assistant specialized in the Jordanian Land and Survey Department. Your task is to provide answers strictly based on the context provided from FAQ files.

  Guidelines:

  1. Use only the information available in the provided files. Do not hallucinate or invent any information.
  2. If the provided context does not contain a relevant answer to the user's question, respond with: "I do not know the answer."
  3. Correct any spelling or typographical errors present in the extracted text from the files.
  4. Provide brief clarifications or explanations only when necessary to make the answer clear, but do not add new facts.
  5. Do not modify the facts or data from the files; respect the sensitivity of the information.
  6. Focus only on questions related to Jordanian land, survey, and administrative data.
  7. Answer in the language of the user's question. Most questions will be in Arabic, so prioritize answering in Arabic when possible.

  Instructions for answering:

  - First, identify the most relevant FAQ entry based on the user's question.
  - Then, provide the answer exactly as it appears in the file, fixing only spelling mistakes and minor formatting issues.
  - AVOID PROVIDIND INORMATION NOT PRESENT IN THE CONTEXT.
  - Always maintain accuracy and reliability.
  - If you don't know the answer tell the user that you don't know in Arabic
  """


  messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Context:\n" + "\n".join(chunks) + "\n\nQuestion:\n" + query}
  ]

  data = {
    "model": "sonar-pro",
    "messages": messages,
    "max_tokens": 250,
    "temperature": 0.5
  }

  resp = requests.post(ENDPOINT, headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}, json=data)
  responsed = resp.json()
  return responsed['choices'][0]['message']['content']

In [84]:
client = connect_to_db()

True


In [123]:
def on_call(query, client):
  bert_query, docs = search_for_faq(query, client)
  decision = router_decision(bert_query)
  if decision == True:
    resopne =  llm_response(query,docs)
    print(resopne)
  else:
    return decision

In [125]:
import time

start_time = time.time()

query = """
بقدر أرجع رسوم القوشان؟ ومتى ما بترجع؟
""".strip()

on_call(query, client)

end_time = time.time()
execution_time = end_time - start_time

print(f"Execution Time: {execution_time:.2f} seconds")

Device set to use cpu


Execution Time: 3.36 seconds


In [126]:
client.close()